In [1]:
from mpramnist.Gosai2024.dataset import GosaiDataset
from mpramnist.Gosai2024.dataset import GosaiDataset_Pairwise
from mpramnist.Gosai2024.trainer import LitModel_Gosai 
from mpramnist.Gosai2024.trainer import LitModel_Gosai_Pairwise

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import torch.nn as nn
import lightning.pytorch as L
from mpramnist import transforms as t

from torch.utils.data import DataLoader

# Testing the Original Dataset Parameters

To use the **exact same parameters** as in the original study, set `filtration = 'original'`. This will apply all filtering criteria from the [**Original Study**](https://www.nature.com/articles/s41586-024-08070-z).

## Key Parameters (Original Settings)

| Parameter               | Value  | Description |
|-------------------------|--------|-------------|
| `stderr_threshold`      | 1.0    | Threshold for the standard error of features; examples above this value are excluded from train/val/test sets |
| `std_multiple_cut`      | 6.0    | Multiplier for standard deviation to define bounds for trusted measurements (removes extreme outliers) |
| `up_cutoff_move`        | 3.0    | Shift factor for upper bound of outlier filter |
| `duplication_cutoff`    | 0.5    | Sequences with max activity higher than this value are duplicated. **Use only during training!** |
| `activity_columns`      | `['K562', 'HepG2', 'SKNSH']` | Column headers containing features to be modeled |
| `stderr`               | `['K562_lfcSE', 'HepG2_lfcSE', 'SKNSH_lfcSE']` | Column headers containing standard errors of features to be modeled |

## Additional Original Settings

- **Sequence padding**: Automatically pads sequences from length 200 to 600 (as done in the original study). Note that padding is performed using the original plasmid sequence rather than 'N' bases.
- **Reverse complements**: Enable with `use_original_reverse_complement = True`. When this parameter is used, the dataset size is doubled.

## Required Transform

For sequence conversion to one-hot, we recommend using `mpramnist.transforms.Seq2Tensor()`

In [2]:
cell_types = ["K562", "HepG2", "SKNSH"]

transform2tensor = t.Compose([t.Seq2Tensor()])

train_dataset_original = GosaiDataset(
    split="train",
    cell_types=cell_types,
    filtration="original",           # use "original" for the authors' parameters
    duplication_cutoff=0.5,          # don't forget to use duplication_cutoff during training
    use_original_reverse_complement=True,  # this parameter pads sequences and adds reverse complements
    transform=transform2tensor,
    root="../data/",
)

val_dataset_original = GosaiDataset(cell_types=cell_types, split="val", filtration="original", transform=transform2tensor, root="../data/")
test_dataset_original = GosaiDataset(cell_types=cell_types, split="test", filtration="original", transform=transform2tensor, root="../data/")

print(train_dataset_original)

Dataset GosaiDataset (MpraDaraset)
    Number of datapoints: 1828680
    Root location: ../data/Malinois
    Using split: ['1', '2', '3', '4', '5', '6', '8', '9', '10', '11', '12', '14', '15', '16', '17', '18', '20', '22', 'Y']
    Split: {'train': 668946, 'val': 58809, 'test': 62582}
    Task: Regression
    Description: The Malinois dataset includes 798,064 sequences tested in the K562, HepG2, and SK-N-SH cell lines. The original sequence length is approximately 200 nucleotides, and it is recommended to extend them to 600 bp. The task is to predict three normalized regulatory activity values for the respective cell lines.


# Using Custom Dataset Parameters

To apply your own filtering criteria, set:  
`filtration = 'own'`

## Custom Transformation Pipeline

Use the following transforms for flexible data processing:

| Transform | Example Usage | Description |
|-----------|---------------|-------------|
| **Padding** | `t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK)` | Adds custom flanking sequences |
| **Cropping** | `t.CenterCrop(600)` | Centers and crops sequences to specified length (600bp) |
| **Reverse Complement** | `t.ReverseComplement(0.5)` | Applies reverse complement transformation with 0.5 probability |

### Key Benefits:
- **Reproduce author methodology**:  
  Combine `AddFlanks` + `CenterCrop` to exactly replicate the original padding approach
- **Memory efficiency**:  
  The `ReverseComplement(0.5)` transform maintains prediction quality while reducing processed dataset size by 50%

In [2]:
cell_types = ["K562", "HepG2", "SKNSH"]

# preprocessing
train_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(0.5), t.Seq2Tensor(),])
val_test_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])

# load the data
train_dataset_own = GosaiDataset(
    split="train",
    transform=train_transform,
    filtration="own",
    cell_types=cell_types,
    stderr_columns=["K562_lfcSE", "HepG2_lfcSE", "SKNSH_lfcSE"],  # change as you want
    stderr_threshold=1.0,  # change as you want
    std_multiple_cut=6.0,  # change as you want
    up_cutoff_move=3.0,  # change as you want
    duplication_cutoff=0.5,  # change as you want
    root="../data/",
)
# Use the same parameters to valid and test
val_dataset_own = GosaiDataset(split="val", filtration="own", cell_types=cell_types, stderr_columns=["K562_lfcSE", "HepG2_lfcSE", "SKNSH_lfcSE"], stderr_threshold=1.0, std_multiple_cut=6.0, up_cutoff_move=3.0, transform=val_test_transform, root="../data/",)
test_dataset_own = GosaiDataset(split="test", filtration="own", cell_types=cell_types, stderr_columns=["K562_lfcSE", "HepG2_lfcSE", "SKNSH_lfcSE"], stderr_threshold=1.0, std_multiple_cut=6.0, up_cutoff_move=3.0, transform=val_test_transform, root="../data/",)

print(train_dataset_own)

Dataset GosaiDataset (MpraDaraset)
    Number of datapoints: 914340
    Root location: ../data/Malinois
    Using split: ['1', '2', '3', '4', '5', '6', '8', '9', '10', '11', '12', '14', '15', '16', '17', '18', '20', '22', 'Y']
    Split: {'train': 668946, 'val': 58809, 'test': 62582}
    Task: Regression
    Description: The Malinois dataset includes 798,064 sequences tested in the K562, HepG2, and SK-N-SH cell lines. The original sequence length is approximately 200 nucleotides, and it is recommended to extend them to 600 bp. The task is to predict three normalized regulatory activity values for the respective cell lines.


# Bypassing Data Filtering

To **disable all filtering and cutoff duplication** and use the **raw** dataset, set:  
`filtration = "none"`

In [4]:
cell_types = ["K562", "HepG2", "SKNSH"]

# preprocessing
train_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK),t.CenterCrop(600),t.ReverseComplement(0.5),t.Seq2Tensor(),])
val_test_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])

train_dataset_filtr_none = GosaiDataset(split="train", cell_types=cell_types, transform=train_transform, filtration="none", root="../data/")
val_dataset_filtr_none = GosaiDataset(split="val", cell_types=cell_types, filtration="none", transform=val_test_transform, root="../data/")
test_dataset_filtr_none = GosaiDataset(split="test", cell_types=cell_types, filtration="none", transform=val_test_transform, root="../data/")

print(train_dataset_filtr_none)

Dataset GosaiDataset (MpraDaraset)
    Number of datapoints: 656560
    Root location: ../data/Malinois
    Using split: ['1', '2', '3', '4', '5', '6', '8', '9', '10', '11', '12', '14', '15', '16', '17', '18', '20', '22', 'Y']
    Split: {'train': 668946, 'val': 58809, 'test': 62582}
    Task: Regression
    Description: The Malinois dataset includes 798,064 sequences tested in the K562, HepG2, and SK-N-SH cell lines. The original sequence length is approximately 200 nucleotides, and it is recommended to extend them to 600 bp. The task is to predict three normalized regulatory activity values for the respective cell lines.


# Training

In [3]:
BATCH_SIZE = 1076
NUM_WORKERS = 103

# encapsulate data into DataLoader form
train_loader = DataLoader(dataset=train_dataset_own, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(dataset=val_dataset_own, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset=test_dataset_own, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset_own[0][0])
out_channels = len(cell_types)

In [4]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Gosai(
    model=model,
    cell_types=cell_types,
    loss=nn.MSELoss(),
    use_one_cycle=True,  # This parameter defines the OneCycleLR scheduler and AdamW optimizer — exactly as required for LegNet architectures.
    weight_decay=0.1,
    lr=0.01,
    print_each=1,
)

In [5]:
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    precision="16-mixed",
    enable_progress_bar=True,
    max_epochs=1,
)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


In [6]:
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Loading `train_dataloader` to estimate number of stepping batches.

  | Name          | Type            | Params | Mode 
----------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | train
1 | loss          | MSELoss         | 0      | train
2 | train_pearson | PearsonCorrCoef | 0      | train
3 | val_pearson   | PearsonCorrCoef | 0      | train
4 | test_pearson  | PearsonCorrCoef | 0      | train
----------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params
5.292     Total estimated model params size (MB)
120       Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  4.18it/s]
------------------------------------------------------------------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 1.79157 
| Val Pearson K562_log2FC: -0.18961 | Val Pearson HepG2_log2FC: -0.19068 | Val Pearson SKNSH_log2FC: 0.25584 | Mean Val Pearson: -0.04148 |
| Train Pearson K562_log2FC: nan | Train Pearson HepG2_log2FC: nan | Train Pearson SKNSH_log2FC: nan | Mean Train Pearson: nan |
------------------------------------------------------------------------------------------------------------------------------------------

                                                                           

/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The ``compute`` method of metric PearsonCorrCoef was called before the ``update`` method which may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The variance of predictions or target is close to zero. This can cause instability in Pearson correlationcoefficient, leading to wrong results. Consider re-scaling the input if possible or computing using alarger dtype (currently using torch.float32). Setting the correlation coefficient to nan.
  warnings.warn(*args, **kwargs)


Epoch 0: 100%|██████████| 850/850 [01:48<00:00,  7.82it/s, v_num=1, train_loss_step=0.535]
-----------------------------------------------------------------------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 0.56182 
| Val Pearson K562_log2FC: 0.83383 | Val Pearson HepG2_log2FC: 0.83716 | Val Pearson SKNSH_log2FC: 0.84271 | Mean Val Pearson: 0.83790 |
| Train Pearson K562_log2FC: 0.69243 | Train Pearson HepG2_log2FC: 0.70398 | Train Pearson SKNSH_log2FC: 0.71803 | Mean Train Pearson: 0.70482 |
-----------------------------------------------------------------------------------------------------------------------------------------------

Epoch 0: 100%|██████████| 850/850 [01:54<00:00,  7.41it/s, v_num=1, train_loss_step=0.535, val_loss=0.562, val_K562_log2FC_pearson=0.834, val_HepG2_log2FC_pearson=0.837, val_SKNSH_log2FC_pearson=0.843, val_pearson=0.838, train_loss_epoch=0.735]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 850/850 [01:54<00:00,  7.40it/s, v_num=1, train_loss_step=0.535, val_loss=0.562, val_K562_log2FC_pearson=0.834, val_HepG2_log2FC_pearson=0.837, val_SKNSH_log2FC_pearson=0.843, val_pearson=0.838, train_loss_epoch=0.735]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 58/58 [00:01<00:00, 29.34it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
test_HepG2_log2FC_pearson   0.7872592210769653
test_K562_log2FC_pearson    0.7679307460784912
test_SKNSH_log2FC_pearson   0.7956514954566956
        test_loss           0.4720252454280853
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.4720252454280853,
  'test_K562_log2FC_pearson': 0.7679307460784912,
  'test_HepG2_log2FC_pearson': 0.7872592210769653,
  'test_SKNSH_log2FC_pearson': 0.7956514954566956}]

# Datasets with A549

By using the `use_A549` parameter, you can obtain a version of the dataset where all sequences have been measured in four cell lines: HepG2, K562, SKNSH, and A549.  
However, this dataset contains only 58k sequences, compared to the full dataset (800k).

In [3]:
BATCH_SIZE = 1076
NUM_WORKERS = 103

cell_types = ["K562", "HepG2", "SKNSH", "A549"]

# preprocessing
train_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(0.5), t.Seq2Tensor(),])
val_test_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])

# load the data
train_dataset_with_A549 = GosaiDataset(
    split="train",
    cell_types=cell_types,
    filtration="original",        
    transform=train_transform,
    use_A549=True,
    root="../data/",
)

val_dataset_with_A549 = GosaiDataset(cell_types=cell_types, split="val", filtration="original", transform=val_test_transform, root="../data/",use_A549=True,)
test_dataset_with_A549 = GosaiDataset(cell_types=cell_types, split="test", filtration="original", transform=val_test_transform, root="../data/",use_A549=True,)

/home/nios/MPRA-MNIST/mpramnist/Gosai2024/dataset.py:226: UserWarning: When use_A549 is set to True, only sequences that have been investigated in all four cell lines (HepG2, K562, SKNSH, A549) are retained, reducing the dataset size
  warnings.warn(
/home/nios/MPRA-MNIST/mpramnist/Gosai2024/dataset.py:226: UserWarning: When use_A549 is set to True, only sequences that have been investigated in all four cell lines (HepG2, K562, SKNSH, A549) are retained, reducing the dataset size
  warnings.warn(
/home/nios/MPRA-MNIST/mpramnist/Gosai2024/dataset.py:226: UserWarning: When use_A549 is set to True, only sequences that have been investigated in all four cell lines (HepG2, K562, SKNSH, A549) are retained, reducing the dataset size
  warnings.warn(


In [4]:
train_loader = DataLoader(dataset=train_dataset_with_A549, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(dataset=val_dataset_with_A549, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset=test_dataset_with_A549, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(len(train_dataset_with_A549), len(val_dataset_with_A549), len(test_dataset_with_A549))

47305 2779 4260


# Pairwise variant effect dataset

In [2]:
BATCH_SIZE = 1076
NUM_WORKERS = 103

cell_types = ["K562", "HepG2", "SKNSH", "GM12878", "A549"]

# preprocessing
train_transform = t.Compose([t.AddFlanks(GosaiDataset_Pairwise.LEFT_FLANK, GosaiDataset_Pairwise.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(0.5), t.Seq2Tensor(),])
val_test_transform = t.Compose([t.AddFlanks(GosaiDataset_Pairwise.LEFT_FLANK, GosaiDataset_Pairwise.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])

# load the data
train_dataset_pairwise = GosaiDataset_Pairwise(
    split="train",
    cell_types=cell_types,
    transform=train_transform,
    root="../data/",
)

val_dataset_pairwise = GosaiDataset_Pairwise(cell_types=cell_types, split="val", transform=val_test_transform, root="../data/")
test_dataset_pairwise = GosaiDataset_Pairwise(cell_types=cell_types, split="test", transform=val_test_transform, root="../data/")

In [14]:
train_loader = DataLoader(dataset=train_dataset_pairwise, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(dataset=val_dataset_pairwise, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset=test_dataset_pairwise, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(len(train_dataset_pairwise), len(val_dataset_pairwise), len(test_dataset_pairwise))

in_channels = len(train_dataset_pairwise[0][0]["seq"])

135041 6623 13619


In [15]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=len(cell_types),
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Gosai_Pairwise(
    model=model,
    cell_types=cell_types,
    loss=nn.MSELoss(),
    use_one_cycle=True,  # This parameter defines the OneCycleLR scheduler and AdamW optimizer — exactly as required for LegNet architectures.
    weight_decay=0.1,
    lr=0.01,
    print_each=1,
)

In [18]:
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    precision="16-mixed",
    enable_progress_bar=True,
    max_epochs=1,
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

SyntaxError: keyword argument repeated: enable_progress_bar (4181752217.py, line 7)

In [19]:
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)

/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:658: Checkpoint directory /home/nios/MPRA-MNIST/examples/Gosai2024_examples/lightning_logs/version_0/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Loading `train_dataloader` to estimate number of stepping batches.

  | Name          | Type            | Params | Mode 
----------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | train
1 | loss          | MSELoss         | 0      | train
2 | train_pearson | PearsonCorrCoef | 0      | train
3 | val_pearson   | PearsonCorrCoef | 0      | train
4 | test_pearson  | PearsonCorrCoef | 0      | train
----------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params
5.294     Total estimated model params size (MB)
120       Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00, 20.45it/s]
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| Epoch: 1 | Val Loss: 0.95653 
| Val Pearson K562_normalized_variant_effect: 0.08745 | Val Pearson HepG2_normalized_variant_effect: 0.08919 | Val Pearson SKNSH_normalized_variant_effect: 0.06809 | Val Pearson GM12878_normalized_variant_effect: 0.03730 | Val Pearson A549_normalized_variant_effect: 0.02655 | Mean Val Pearson: 0.06172 |
| Train Pearson K562_normalized_variant_effect: nan | Train Pearson HepG2_normalized_variant_effect: nan | Train Pearson SKNSH_normalized_variant_effect: nan | Train Pearson GM12878_normalized_variant_effect: nan | Train Pearson A549_normalized_variant_effect: nan | Mean Train Pears

/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The ``compute`` method of metric PearsonCorrCoef was called before the ``update`` method which may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The variance of predictions or target is close to zero. This can cause instability in Pearson correlationcoefficient, leading to wrong results. Consider re-scaling the input if possible or computing using alarger dtype (currently using torch.float32). Setting the correlation coefficient to nan.
  warnings.warn(*args, **kwargs)
`Trainer.fit` stopped: `max_epochs=1` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 15.48it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                 Test metric                                   DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 test_A549_normalized_variant_effect_pearson               0.032121527940034866
test_GM12878_normalized_variant_effect_pearson            0.0014941152185201645
 test_HepG2_normalized_variant_effect_pearson              0.044314365833997726
 test_K562_normalized_variant_effect_pearson               0.056044090539216995
 test_SKNSH_normalized_variant_effect_pearson              0.06278488039970398
                  test_loss                                 0.9513084888458252
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.9513084888458252,
  'test_K562_normalized_variant_effect_pearson': 0.056044090539216995,
  'test_HepG2_normalized_variant_effect_pearson': 0.044314365833997726,
  'test_SKNSH_normalized_variant_effect_pearson': 0.06278488039970398,
  'test_GM12878_normalized_variant_effect_pearson': 0.0014941152185201645,
  'test_A549_normalized_variant_effect_pearson': 0.032121527940034866}]